Libraries imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

Read and transformations

In [0]:
df_sales = spark.table("abinbev_case_silver.fact_sales")

df_region_brand = (
    df_sales
    .groupBy("btlr_org_lvl_c_desc", "brand_nm")
    .agg(F.sum("volume").alias("total_volume"))
)

window_lowest = Window.partitionBy("btlr_org_lvl_c_desc").orderBy(F.asc("total_volume"))

df_lowest_brand_by_region = (
    df_region_brand
    .withColumn("rank", F.dense_rank().over(window_lowest))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("btlr_org_lvl_c_desc")
)

Save as gold delta table

In [0]:
df_lowest_brand_by_region.write.mode("overwrite").format("delta").saveAsTable(
    "abinbev_case_gold.v_lowest_brand_by_region"
)

display(df_lowest_brand_by_region)